# Ungraded Lab - Introduction to Weaviate API
# 未分级实验室- Weaviate API介绍


<div align="center">
  <img src="images/weaviate.png" alt="RAG Overview" width="10%">
</div>

Welcome to the ungraded lab about the Weaviate API! As you dive into the world of vector databases, you'll discover that there are several options available to help you build your RAG systems. In this course, you'll focus on the [Weaviate API](https://weaviate.io/).

欢迎来到Weaviate API的未分级实验室！当您深入到矢量数据库的世界中时，您会发现有几个选项可以帮助您构建RAG系统。在本课程中，您将专注于[Weaviate API](https://weaviate.io/)。

This lab is designed to give you a hands-on introduction to Weaviate, so you'll be well-prepared for the upcoming assignment. You'll explore how Weaviate functions, what it can do, and how to make the most of its features. By the time you reach the assignment, you'll have the tools and knowledge you need to succeed.

这个实验的目的是给你一个动手介绍Weaviate，所以你会为即将到来的任务做好充分的准备。您将探索如何使用Weaviate函数，它可以做什么，以及如何充分利用它的特性。当你完成任务时，你就拥有了成功所需的工具和知识。

Let's go!



# 目录 (Table of Contents)
- [ 1 - 简介 (Introduction)](#1)
  - [ 1.1 加载必要的代码库 (Loading the necessary libraries)](#1-1)
  - [ 1.2 Weaviate 客户端 (The Weaviate Client)](#1-2)
- [ 2 - 配置数据库 (Configuring the database)](#2)
  - [ 2.1 创建集合 (Creating a Collection)](#2-1)
  - [ 2.2 配置向量化模型 (Configuring the Vectorizer)](#2-2)
  - [ 2.3 属性字段 (The Properties)](#2-3)
  - [ 2.4 向集合中添加元素 (Adding elements into a Collection)](#2-4)
- [ 3 - 在集合中进行查询 (Querying on a collection)](#3)
  - [ 3.1 过滤器/条件过滤 (Filters)](#3-1)
  - [ 3.2 语义搜索 (Semantic Search)](#3-2)
  - [ 3.3 BM25 搜索 (BM25 search)](#3-3)
  - [ 3.4 混合搜索 (Hybrid Search)](#3-4)
  - [ 3.5 结果重排 (Reranking)](#3-5)

---
<h4 style="color:black; font-weight:bold;">USING THE TABLE OF CONTENTS</h4>
<h4 style="color:black; font-weight:bold;">使用目录表</h4>

JupyterLab provides an easy way for you to navigate through your assignment. It's located under the Table of Contents tab, found in the left panel, as shown in the picture below.

JupyterLab为您提供了一种简单的方法来浏览您的作业。它位于左侧面板的目录选项卡下，如下图所示。

![TOC Location](images/toc.png)

---

<a id='1'></a>
## 1 - Introduction
---
<a id='1-1'></a>
### 1.1 Loading the necessary libraries
Run the cell below to load the necesary libraries for this assignment.

### 1.1 加载必要的库
运行下面的单元格以加载此任务所需的库。

In [1]:
# 从 Weaviate 配置模块导入核心工具
from weaviate.classes.config import Configure, Property, DataType
# 作用：用于定义集合（Collection）的结构。
# - Configure: 配置向量化器、索引参数等。
# - Property: 定义数据字段（如“标题”、“内容”）。
# - DataType: 指定字段类型（如文本 TEXT、整数 INT、布尔值 BOOL 等）。

# 导入查询过滤器
from weaviate.classes.query import Filter
# 作用：用于在搜索时执行属性过滤。
# 例如：“在向量搜索的同时，只展示分类为‘科技’的文章”。

from typing import List  # 导入类型提示，用于声明列表类型
from tqdm import tqdm  # 导入进度条库，在批量导入成千上万条数据时提供视觉反馈

import joblib  # 导入对象序列化库，用于加载之前保存的 .joblib 数据文件
import weaviate  # 导入 Weaviate 客户端主库
import re  # 导入正则表达式库，用于在导入前对原始文本进行清洗或匹配

from weaviate.util import generate_uuid5  # 导入 UUID5 生成器
# 作用：根据内容生成确定性的唯一标识符（UUID）。
# 核心价值：防止数据重复导入。如果内容相同，生成的 UUID 必相同，Weaviate 会执行更新而非新增。

from pprint import pprint  # 导入“美化打印”函数，让复杂的嵌套查询结果在控制台更易读
import os  # 导入操作系统接口，用于处理文件路径和环境变量

In [ ]:
!pip install Flask

In [ ]:
!pip install FlagEmbedding

In [ ]:
!pip install transformers==4.44.2

In [ ]:
# 从自定义的工具包 utils 中导入必要的辅助函数
from utils import (
    suppress_subprocess_output, # 用于静默子进程输出的上下文管理器
    generate_with_single_input, # 用于调用大模型（LLM）生成回答的函数
    print_object_properties,    # 用于美化打印对象属性的工具
    kill_processes_on_ports     # 核心工具：用于强制杀掉占用特定端口的进程
)

# 在导入 flask_app 之前，先清理指定的端口
# 警告：如果你的 Jupyter 内核或当前服务正运行在这些端口上，运行此单元格可能会导致连接断开或内核重启
# kill_processes_on_ports([5000, 8080, 8097, 50050, 50051])
# 作用：一次性清理 5000 (Flask), 8080 (Weaviate API), 50050/50051 (gRPC) 等常用端口。
# 这样可以保证后续启动服务时，不会因为“Address already in use”（地址已被占用）而崩溃。
os.environ["MODEL_M3"] = os.path.expanduser("~/Desktop/AIAgent/models")
import flask_app
# 作用：正式导入并启动 Flask 应用程序逻辑。
# 由于上面已经清理了端口，这里的导入和初始化通常会非常顺畅。

<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


 * Serving Flask app 'flask_app'
 * Debug mode: off


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


<a id='1-2'></a>
### 1.2 - The Weaviate Client

To start working with the Weaviate API in this environment, you need to start a `client`. In this course, you will use an *embedded client*, which is a way of using Weaviate within this application and not relying on a stand-alone instance of Weaviate running. 

When you start `Embedded Weaviate` for the first time, it creates a data storage file at the location you specify in `persistence_data_path`. Even after you close your client and Embedded Weaviate shuts down, your data will still be saved there. 

When creating your Weaviate client, you must pass an embedding model to perform vectorization. You can pass different models and Weaviate has different modules to help you call `OpenAI` models and others. Since OpenAI is a paid system, we will use a local model to perform the vectorization. 

One example to load an OpenAI model is this call:

### 1.2 -编织客户端
要在此环境中开始使用Weaviate API，您需要启动一个“客户端”。在本课程中，您将使用*嵌入式客户端*，这是在此应用程序中使用Weaviate的一种方式，而不依赖于Weaviate运行的独立实例。

当你第一次启动`Embedded Weaviate`时，它会在你在`persistence_data_path`中指定的位置创建一个数据存储文件。即使在您关闭客户端和Embedded Weaviate关闭后，您的数据仍将保存在那里。

在创建Weaviate客户端时，必须传递一个嵌入模型来执行向量化。你可以传递不同的模型，Weaviate有不同的模块来帮助你调用“OpenAI”模型和其他。由于OpenAI是一个付费系统，我们将使用一个本地模型来执行矢量化。

加载OpenAI模型的一个例子是这样的调用：

```Python
import weaviate  # 导入 Weaviate 客户端库，用于操作向量数据库

# 使用嵌入式连接方式启动并连接 Weaviate
# 这种模式不需要你手动运行 Docker，Weaviate 会在后台自动下载并运行二进制文件
client = weaviate.connect_to_embedded(
    version="1.26.1",  # 指定要运行的 Weaviate 核心版本号
    
    # 设置全局请求头
    headers={
        # 提供 OpenAI 的 API 密钥
        # 作用：当你在 Weaviate 中使用 'text2vec-openai' 模块时，
        # 数据库会自动使用这个密钥调用 OpenAI 的接口来生成向量。
        "X-OpenAI-Api-Key": "你的_OPENAI_API_密钥" 
    },
)
```

Let's load our client! 

让我们加载我们的客户端！

This function `suppress_subprocess_output()` is designed to suppress the Weaviate output logs, which can pollute the lab. These logs won't be explored in this lab, but feel free to remove it if you are curious about what the logs look like! The arguments it will be using are:

这个函数`suppress_subprocess_output()`被设计用来抑制Weaviate输出日志，因为它可能会污染实验室。这些日志将不会在本实验中进行探索，但是如果您对日志的外观感到好奇，可以随意删除它！它将使用的参数是：

- `persistence_data_path`: The path where the client will look for (and create) the vector databases. Once you create it, it is stored there and persisted, i.e., it won't be deleted once you close the client!
- `environment_vabiables`: Necessary variables that we must pass to make the local embedding server to work. 

- `persistence_data_path`：客户端将查找（并创建）矢量数据库的路径。一旦您创建了它，它就存储在那里并持久化，也就是说，一旦您关闭客户端，它就不会被删除！
- `environment_vabables`：我们必须传递的必要变量，以使本地嵌入服务器工作。

In [3]:
# 使用之前定义的上下文管理器，在启动过程中静默所有子进程产生的冗长系统日志
with suppress_subprocess_output():
# 启动并连接到一个嵌入式 Weaviate 实例（无需手动开启外部数据库服务）
    client = weaviate.connect_to_embedded(
        # 指定数据持久化路径：所有创建的集合和存入的文档都会保存在当前目录下的 .collections 文件夹中
        # 这样即使你关闭程序，下次启动时数据依然存在
        persistence_data_path="./.collections", 
        
        # 配置 Weaviate 实例的环境变量，用于自定义其 AI 模块的行为
        environment_variables={
            # 开启基于 API 的外部模块支持，允许 Weaviate 调用外部接口处理数据
            "ENABLE_API_BASED_MODULES": "true", 
            
            # 启用具体的 AI 模块：
            # 1. text2vec-transformers: 用于将文本转化为向量
            # 2. reranker-transformers: 用于对检索结果进行精细化的二次重排序
            "ENABLE_MODULES": 'text2vec-transformers, reranker-transformers', 
            
            # 核心向量化配置：指定 Weaviate 执行“文本转向量”任务时访问的 API 端点
            # 这里的 5000 端口通常对应你之前启动的 Flask 向量化微服务
            "TRANSFORMERS_INFERENCE_API": "http://127.0.0.1:5000/", 
            
            # 核心重排序配置：指定 Weaviate 执行“结果重排序”任务时访问的 API 端点
            # 同样指向你本地运行的 Flask 服务（该服务内集成了 BGE-Reranker 模型）
            "RERANKER_INFERENCE_API": "http://127.0.0.1:5000/" 
        }
    )

With the defined `client`, your primary usage is creating a collection, adding elements to it and querying over it.

使用定义的“客户端”，您的主要用途是创建一个集合，向其添加元素并对其进行查询。

<a id='2'></a>
## 2 - Configuring the database

## 2 -配置数据库
---

In this section, you will explore the central object in this lab and in this assignment: [the collection](https://weaviate.io/developers/weaviate/manage-data/collections) - this is the name Weaviate gives to a group of data objects which will be indexed for retrieval. Remember the workflow from the lectures:

在本节中，您将探索本实验和作业中的中心对象：[the collection](https://weaviate.io/developers/weaviate/manage-data/collections)——这是weaviate为一组数据对象提供的名称，这些数据对象将被索引以供检索。还记得讲座上的工作流程吗：

<div align="center">
  <img src="images/workflow.png" alt="RAG Overview" width="60%">
</div>


<a id='2-1'></a>
### 2.1 Creating a Collection
### 2.1 创建集合

To create a collection, there are some parameters that must be set. The most important for our purposes are:

- `name`: the collection name, this is the name that will be saved in memory and the name that you will need to load it.
- `vectorizer_config`: a list with vectorizer configurations. You can pass more than one vectorizer configuration, which means that in the same vector database, you can vectorize your datapoints with different embedding models. In your context, you will be using only one.

Let's load a database to illustrate this section.

要创建集合，必须设置一些参数。对于我们的目的来说，最重要的是：

- `name`：集合名称，这是将保存在内存中的名称，您将需要加载它。
- `vectorizer_config`：一个矢量器配置列表。您可以传递多个矢量化器配置，这意味着在同一个矢量数据库中，您可以使用不同的嵌入模型对数据点进行矢量化。在您的上下文中，您将只使用一个。

让我们加载一个数据库来说明这一部分。

In [4]:
# 从硬盘加载序列化的数据文件
data = joblib.load("data.joblib")
# 作用：使用 joblib 库将之前保存的二进制文件（data.joblib）还原为 Python 对象。
# 在你的项目中，这个文件通常包含了一个由字典组成的列表，
# 里面存储了文章标题、内容分块（chunks）以及对应的分类信息。

# 使用自定义工具函数预览第一条数据
print_object_properties(data[0])
# 作用：调用你之前定义的 print_object_properties 函数，
# 提取并打印数据列表中第一个元素（索引为 0）的所有属性。
# 该函数会自动截断超长字段（如正文或高维向量），让你能直观、整洁地确认数据加载是否成功。

place: Grand Canyon
state: Arizona
description: A stunning canyon with vast vistas and incredible geology.
best_season_to_visit: Spring, Fall
attractions: South Rim, Havasu Falls, Skywalk
budget: Moderate
user_ratings: 4.8
last_updated: 2023-10-01T00:00:00Z





The dataset is a set of places to visit, with some properties describing each location. The properties here are `place, state, description, best_season_to_visit, attractions, budget, user_ratings, last_updated`. When creating a collection, you must create one property for each key in this dictionary and add the expected datatype. 

数据集是一组要访问的地点，每个地点都有一些属性来描述。这里的属性是`place, state, description, best_season_to_visit, attractions, budget, user_ratings, last_updated`。创建集合时，必须为该字典中的每个键创建一个属性，并添加预期的数据类型。

<a id='2-2'></a>
### 2.2 配置向量器

如前所述，你将使用 `text2vec_transformers` 嵌入模型对数据进行向量化。要对其进行配置，你必须传递相应的 `Configure` 对象。在配置向量器时，你可以传递一个包含不同向量器的列表，这样你的集合就可以为同一个对象存储多种向量化表示。你还可以选择在特定的向量器上对特定的属性进行向量化。在本课程中，你将坚持使用一种向量器。并非每个属性都必须被向量化，这取决于数据和你想要检索的信息。

在这种情况下，让我们对以下属性进行向量化：
`place` (地点), `state` (州/省), `description` (描述), `best_season_to_visit` (最佳旅游季节), `attractions` (景点), `budget` (预算)

这些属性将相互拼接，然后进行向量化。在定义属性时，你可以选择是否在向量化中包含属性名称。请注意，在 `budget` 中包含属性名称是有意义的，例如，仅凭 "Moderate"（中等）一词无法提供关于 "Moderate" 代表什么的充足信息。

In [5]:
# 定义向量化器的配置列表（Weaviate 支持为一个对象生成多个不同的向量）
vectorizer_config = [
    # 使用 NamedVectors 模式下的 text2vec-transformers 插件进行配置
    Configure.NamedVectors.text2vec_transformers(
        # 向量的名称：后续你在进行向量搜索（如 nearVector）时，需要通过这个名称来调用
        name="vector", 
        
        # 属性源：指定哪些字段的数据应该被用来生成向量。
        # 数据库会将这些字段的内容拼接在一起发送给模型。例如：将“地点”、“描述”和“预算”合并理解。
        source_properties=['place', 'state', 'description', 'best_season_to_visit', 'attractions', 'budget'], 
        
        # 向量化集合名称：设置为 False 表示在生成向量时，不包含集合（表）的名字。
        # 如果设为 True，集合名（如 "TravelDestinations"）会被添加在所有文本的最前面。
        vectorize_collection_name = False, 
        
        # 推理接口地址：由于我们使用的是基于 API 的向量化器，必须传入执行计算的 URL。
        # 这里的 5000 端口正是对应我们之前用 Flask 搭建的那个加载了 BGE 模型的服务。
        inference_url="http://127.0.0.1:5000", 
    )
]

<a id='2-3'></a>
### 2.3 The Properties
### 2.3 属性

In a collection, the features of each data point are called Properties.

在集合中，每个数据点的特征称为属性。

In [ ]:
# 检查集合是否存在，如果存在则将其删除，以便重新开始
if client.collections.exists("example_collection"):
    # 作用：调用 Weaviate 客户端检查数据库中是否已经有名为 "example_collectiom" 的集合。
    # 注意：这里的代码里有一个小小的拼写错误（typo），最后是 'm' 而不是 'n'。

    client.collections.delete("example_collection")
    # 作用：如果上述检查返回 True（集合存在），则执行物理删除操作。
    # 警告：删除集合会同时清空该集合下的所有文档、向量和索引，操作不可逆。
    # 注意：这里你写的是 "example_collection"（结尾是 'n'），
    # 建议将两行的拼写统一，否则可能因为逻辑不匹配导致删除失败。

In [7]:
# 检查数据库中是否已有名为 'example_collection' 的集合，不存在时才创建
if not client.collections.exists('example_collection'): 
    # 正式创建集合
    collection = client.collections.create(
            name='example_collection', # 集合名称
            
            # 向量化配置：使用我们之前定义的 vectorizer_config（指定了哪些字段参与建模，以及 API 地址）
            vectorizer_config=vectorizer_config, 
            
            # 重排序配置：启用 transformers 重排序模块
            # 作用：在搜索时，Weaviate 会自动调用之前在环境配置中指定的 RERANKER_INFERENCE_API
            reranker_config=Configure.Reranker.transformers(), 

            # 定义属性（字段）列表
            properties=[  
                # Property 参数解析：
                # vectorize_property_name=True 表示在生成向量时，把“字段名”也加进去（例如：文本中包含 "place: ..."）
                # data_type 指定了存储的数据类型
                
                Property(name="place", vectorize_property_name=True, data_type=DataType.TEXT),
                Property(name="state", vectorize_property_name=True, data_type=DataType.TEXT),
                Property(name="description", vectorize_property_name=True, data_type=DataType.TEXT),
                Property(name="best_season_to_visit", vectorize_property_name=True, data_type=DataType.TEXT),
                Property(name="attractions", vectorize_property_name=True, data_type=DataType.TEXT),
                Property(name="budget", vectorize_property_name=True, data_type=DataType.TEXT),
                
                # 数字类型：不参与向量化建模，但可以用于过滤（例如：筛选评分 > 4.5 的地点）
                Property(name="user_ratings", data_type=DataType.NUMBER),
                
                # 日期类型：用于时间维度的查询或排序
                Property(name="last_updated", data_type=DataType.DATE),
            ]
        )
else:
    # 如果集合已经存在，则直接获取该集合的操作句柄
    collection = client.collections.get("example_collection")

/Users/a1-6/miniconda3/envs/ailearn/lib/python3.11/site-packages/weaviate/warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


Running it creates a collection and returns the collection. Printing it shows the collection configuration.

运行它将创建一个集合并返回该集合。打印它将显示集合配置。

In [8]:
print(collection)

<weaviate.Collection config={
  "name": "Example_collection",
  "description": null,
  "generative_config": null,
  "inverted_index_config": {
    "bm25": {
      "b": 0.75,
      "k1": 1.2
    },
    "cleanup_interval_seconds": 60,
    "index_null_state": false,
    "index_property_length": false,
    "index_timestamps": false,
    "stopwords": {
      "preset": "en",
      "additions": null,
      "removals": null
    }
  },
  "multi_tenancy_config": {
    "enabled": false,
    "auto_tenant_creation": false,
    "auto_tenant_activation": false
  },
  "object_ttl_config": null,
  "properties": [
    {
      "name": "place",
      "description": null,
      "data_type": "text",
      "index_filterable": true,
      "index_range_filters": false,
      "index_searchable": true,
      "nested_properties": null,
      "tokenization": "word",
      "vectorizer_config": null,
      "vectorizer": null,
      "vectorizer_configs": {
        "text2vec-transformers": {
          "skip": false,
 

If you try to create a collection that already exists, an exception will be thrown:

如果你尝试创建一个已经存在的集合，将会抛出一个异常：

In [9]:
try:
    # 尝试在 Weaviate 中创建一个新的集合（Collection）
    collection = client.collections.create(
        # 集合的名称，相当于关系型数据库中的“表名”
        name='example_collection',

        # 绑定向量化配置：决定了数据存入时如何转化为数字向量
        vectorizer_config=vectorizer_config, 
    
        # 定义属性（字段）列表，确定该集合能够存储哪些信息
        properties=[  
            # Property 参数详解：
            # name: 字段名称
            # vectorize_property_name=True: 告诉模型在向量化时包含字段名，增加语义上下文
            # data_type: 指定数据类型（如文本 TEXT、数字 NUMBER、日期 DATE）

            Property(name="place", vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="state", vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="description", vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="best_season_to_visit", vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="attractions", vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="budget", vectorize_property_name=True, data_type=DataType.TEXT),
            
            # 以下两个字段仅存储原始数据，不参与“语义向量”的生成计算
            Property(name="user_ratings", data_type=DataType.NUMBER), # 用户评分
            Property(name="last_updated", data_type=DataType.DATE),    # 最后更新时间
        ]
    )
# 如果创建过程中出现任何错误（例如：集合已存在、配置参数非法、网络连接超时等）
except Exception as e:
    # 打印具体的错误信息，方便开发者排查问题
    print(e)

Collection may not have been created properly.! Unexpected status code: 422, with response body: {'error': [{'message': 'class name Example_collection already exists'}]}.


You can also retrieve all the collections saved:

In [10]:
client.collections.list_all().keys()
# 作用：获取当前 Weaviate 实例中所有已创建集合（Collection）的名称列表。
# client.collections.list_all()：返回一个字典（或类字典对象），包含数据库中所有集合的详细配置信息。
# .keys()：从这个字典中仅提取出所有集合的名字（键名）。
# 输出示例：dict_keys(['Faq', 'Article', 'example_collection'])

dict_keys(['Example_collection'])

The result of .list_all() is a dictionary with the collections names as keys and their properties.

的结果.List_all（）是一个以集合名称作为键及其属性的字典。

<a id='2-4'></a>
### 2.4 向集合中添加元素

一旦创建了集合，你得到的是一个空集合。现在你需要向其中添加元素。当你添加一个元素时，后台会发生两个重要的步骤：

1. **信息被向量化**（按照集合定义中的配置）。
2. **HNSW 索引被更新**以优化搜索（正如你在课程中所见）。这发生在后台，你无法直接看到，但这可能会使处理过程耗费一些时间。

添加元素是通过 `collection.batch` 完成的，它提供了一些额外的实用功能。例如，它允许你决定每个批次发送的对象数量、处理导入过程中的错误，并通过减少单个网络调用的次数来提升性能。在本示例中，一次添加一个元素，且每次只有一个并发请求。

你可以为添加的每个元素分配一个 **UUID**（唯一标识符 ID），这可以防止数据库中出现重复记录。

让我们来看一下实际操作！

In [11]:
# 设置一个固定大小和并发数的批量处理上下文管理器
with collection.batch.fixed_size(batch_size=1, concurrent_requests=1) as batch:
    # 作用：开启批量导入模式。
    # batch_size=1: 每次积攒 1 个对象就发送给服务器（通常生产环境会设为 100+ 以提升效率）。
    # concurrent_requests=1: 设置并发请求数为 1，即按顺序一个一个处理。
    # 使用 'with' 语句能确保在导入结束时，所有残留的缓存数据都能正确提交并关闭连接。

    # 遍历数据集，并使用 tqdm 显示实时进度条
    for document in tqdm(data): 
        # tqdm 作用：在终端显示一个类似 [###-------] 20% 的进度条，让你知道导入还需多久。

        # 基于文档内容生成一个 UUID5（版本 5 唯一标识符）
        # 作用：这是一种“确定性 UUID”。只要 document 的内容不变，生成的 UUID 永远相同。
        # 核心价值：防止数据重复。如果重复运行此脚本，Weaviate 会识别出 ID 已存在，从而执行“更新”而非“重复插入”。
        uuid = generate_uuid5(document)

        # 将对象添加到当前批次中
        batch.add_object(
            # properties: 传入包含字段名和内容的字典（如地名、描述、评分等）。
            properties=document,
            # uuid: 传入刚才生成的指纹 ID。
            uuid=uuid,
        )

100%|██████████| 20/20 [00:02<00:00,  9.83it/s]


Awesome! Now you have a collection with vectors! You can check the number of vectors using `len(collection)`:

太棒了!现在你有了一个向量集合！你可以使用`len(collection)`来检查向量的数量：

In [12]:
len(collection)

20

<a id='3'></a>
## 3 - Querying on a collection
## 3 -查询一个集合

In this section, you will learn how to query on a collection. You can:

- Query on metadata
- Query with semantic search
- Query with BM25
- Query with filtering

Let's see some examples.

在本节中，您将学习如何查询集合。您可以:
- 元数据查询
- 使用语义搜索查询
- 通过BM25查询
- 带过滤的查询

让我们看一些例子。

<a id='3-1'></a>
### 3.1 Filters
### 3.1过滤器

Before diving into querying, let's understand the Filters. Filters are a way of restricting your search on some criteria. They are very flexible. You usually pass them as an argument in a query. Let's have an example to illustrate it.

在深入查询之前，让我们先了解一下Filters。过滤器是一种将搜索限制在某些条件上的方法。它们非常灵活。通常在查询中将它们作为参数传递。让我们举个例子来说明。

In [13]:
# 获取 2 个对象，并添加属性过滤器：筛选出 'user_ratings' 大于或等于 3.5 的对象
result = collection.query.fetch_objects(
    # limit: 限制返回结果的数量。这里只取前 2 条匹配的数据。
    limit = 2, 
    
    # filters: 定义过滤逻辑。
    # Filter.by_property('user_ratings'): 指定要过滤的字段名为 'user_ratings'。
    # .greater_or_equal(3.5): 设置筛选条件为数值大于或等于 3.5。
    filters = Filter.by_property('user_ratings').greater_or_equal(3.5)
)

The result is an object called QueryReturn:

结果是一个名为QueryReturn的对象：

In [14]:
result

QueryReturn(objects=[Object(uuid=_WeaviateUUIDInt('c99763a3-46a0-59d4-831b-af9bc290260c'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'place': 'Hollywood', 'best_season_to_visit': 'Spring', 'last_updated': datetime.datetime(2023, 10, 1, 0, 0, tzinfo=datetime.timezone.utc), 'description': 'Famous district in Los Angeles known as the entertainment capital of the world.', 'state': 'California', 'attractions': 'Walk of Fame, Hollywood Sign', 'budget': 'Moderate', 'user_ratings': 4.2}, references=None, vector={}, collection='Example_collection'), Object(uuid=_WeaviateUUIDInt('9e5ba590-8c75-5b53-9b0a-8a9c161004ad'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'place': 'Times Square', 'best_season_to_visit': 'Winter', 'l

You can access its objects by `result.objects`

你可以通过`result.objects`访问它的对象

In [15]:
result.objects

[Object(uuid=_WeaviateUUIDInt('c99763a3-46a0-59d4-831b-af9bc290260c'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'place': 'Hollywood', 'best_season_to_visit': 'Spring', 'last_updated': datetime.datetime(2023, 10, 1, 0, 0, tzinfo=datetime.timezone.utc), 'description': 'Famous district in Los Angeles known as the entertainment capital of the world.', 'state': 'California', 'attractions': 'Walk of Fame, Hollywood Sign', 'budget': 'Moderate', 'user_ratings': 4.2}, references=None, vector={}, collection='Example_collection'),
 Object(uuid=_WeaviateUUIDInt('9e5ba590-8c75-5b53-9b0a-8a9c161004ad'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'place': 'Times Square', 'best_season_to_visit': 'Winter', 'last_updated': datet

So, each element in the list is an element of the collection. 

因此，列表中的每个元素都是集合中的一个元素。

In [16]:
obj = result.objects[0]

You can check their properties, which is a dictionary.

你可以检查它们的属性，这是一个字典。

In [17]:
obj.properties

{'place': 'Hollywood',
 'best_season_to_visit': 'Spring',
 'last_updated': datetime.datetime(2023, 10, 1, 0, 0, tzinfo=datetime.timezone.utc),
 'description': 'Famous district in Los Angeles known as the entertainment capital of the world.',
 'state': 'California',
 'attractions': 'Walk of Fame, Hollywood Sign',
 'budget': 'Moderate',
 'user_ratings': 4.2}

In this course, the way of filtering is `.by_property`. You will see more Filtering examples as other query methods are explained.

在本课程中，过滤的方式是`.by_property`。随着其他查询方法的介绍，您将看到更多的Filtering示例。

<a id='3-2'></a>
### 3.2 Semantic Search
### 3.2 语义搜索

You can use semantic search to query over your collection. This uses the vectors to compute distances between them and return the closest ones. You must pass a query, which will be vectorized and then compared over the elements on your collection. The method is `.near_text`.

您可以使用语义搜索来查询您的集合。它使用向量来计算它们之间的距离，并返回最近的距离。您必须传递一个查询，该查询将被向量化，然后与集合上的元素进行比较。方法是 `.near_text`。

In [18]:
# 执行语义搜索（向量搜索）：寻找与用户描述的“意图”在含义上最接近的 4 个结果
result = collection.query.near_text(
    # query: 用户的自然语言输入。
    # 这里的意图是：想在“冬天”旅游，且要求“便宜”。
    query = 'I want suggestions to travel during Winter. I want cheap places.', 
    
    # limit: 限制返回结果的数量，这里只取出语义最匹配的前 4 个景点。
    limit = 4
)

In [19]:
# 遍历查询结果中的每一个对象，并打印它们的属性
for obj in result.objects:
    # result.objects 是一个包含所有匹配文档的列表
    # obj 是列表中的一个元素，代表一个搜索到的地点信息
    
    # 调用自定义的 print_object_properties 函数
    # obj.properties 是一个字典，包含了我们在创建集合时定义的所有字段（如 place, budget 等）
    print_object_properties(obj.properties)
    # 作用：将这个字典里的内容整齐地打印出来，并对长文本进行截断处理

place: Times Square
best_season_to_visit: Winter
last_updated: 2023-10-01 00:00:00+00:00
description: Bustling pedestrian intersection and major commercial hub.
state: New York
attractions: Broadway Theaters, New Year’s Eve Ball Drop
budget: Low
user_ratings: 4.3



place: Glacier National Park
state: Montana
last_updated: 2023-10-01 00:00:00+00:00
description: Park known for its rugged mountains and alpine forests.
best_season_to_visit: Summer
attractions: Going-to-the-Sun Road, Grinnell Glacier
budget: Moderate
user_ratings: 4.8



last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Spring, Fall
place: Zion National Park
description: Beautiful park known for its impressive canyons and towering cliffs.
state: Utah
attractions: The Narrows, Angels Landing
budget: Moderate
user_ratings: 4.7



place: Cape Cod
best_season_to_visit: Summer
last_updated: 2023-10-01 00:00:00+00:00
description: Popular tourist destination known for its beaches and quaint towns.
state: Massachusetts

You can also already query over the elements with `budget = Low`:

你也可以用`budget = Low`来查询元素：

In [20]:
# 执行“语义搜索 + 属性过滤”的复合查询
result = collection.query.near_text(
    # query: 语义搜索的自然语言描述
    # 模型会理解“Winter”和“Travel”的含义，去寻找相关的地点描述
    query = 'I want suggestions to travel during Winter. I want cheap places.', 
    
    # filters: 硬性过滤条件（元数据过滤）
    # 作用：在进行语义匹配之前或同时，强制要求 'budget' 字段的值必须等于 'Low'。
    # 这能确保即便模型认为某个“中等价位”的景点很相关，也会被系统直接剔除。
    filters = Filter.by_property('budget').equal('Low'),
    
    # limit: 限制最终返回的结果数量为 4 条
    limit = 4
)

In [21]:
# 遍历查询结果中的每一个匹配对象
for obj in result.objects:
    # result.objects 是 Weaviate 返回的列表，按匹配程度（相似度）从高到低排序
    # 每个 obj 代表一个具体的数据库记录（在本例中是一个旅游景点）
    
    # 调用之前定义的自定义函数 print_object_properties 来展示数据
    # obj.properties：这是一个字典，包含了该景点的所有详细信息（如地名、描述、评分等）
    print_object_properties(obj.properties)
    # 该函数会自动格式化输出，并对过长的文本块（chunk）进行截断，让控制台看起来更整洁

place: Times Square
best_season_to_visit: Winter
last_updated: 2023-10-01 00:00:00+00:00
description: Bustling pedestrian intersection and major commercial hub.
state: New York
attractions: Broadway Theaters, New Year’s Eve Ball Drop
budget: Low
user_ratings: 4.3



last_updated: 2023-10-01 00:00:00+00:00
state: California
place: Alcatraz Island
description: Famed former prison island located in San Francisco Bay.
best_season_to_visit: Spring, Summer
attractions: Cellhouse Tour, Alcatraz Lighthouse
budget: Low
user_ratings: 4.4



place: Gettysburg National Military Park
state: Pennsylvania
last_updated: 2023-10-01 00:00:00+00:00
description: Historic site of a major Civil War battle.
best_season_to_visit: Spring, Fall
attractions: Gettysburg Museum, Battlefield Tours
budget: Low
user_ratings: 4.6



last_updated: 2023-10-01 00:00:00+00:00
state: New York
place: Statue of Liberty
description: Iconic symbol of freedom and democracy in the United States.
best_season_to_visit: Spring, Fal

You can also pass a list on the possible values on a filter, by using `.contains_any`:

你也可以通过使用`.contains_any`来传递过滤器上可能值的列表：

In [22]:
# 执行“语义搜索 + 多值逻辑过滤”的复合查询
result = collection.query.near_text(
    # query: 语义搜索的自然语言描述
    # 系统会寻找关于“冬季旅游”和“省钱”语义相关的景点
    query = 'I want suggestions to travel during Winter. I want cheap places.', 
    
    # filters: 属性过滤器（使用 contains_any 逻辑）
    # 作用：筛选出 'budget' 字段属于 ['Low', 'Moderate'] 列表内任一值的对象。
    # 相比 .equal('Low')，这种方法扩大了筛选范围，允许搜索“低预算”或“中等预算”的地点。
    # 类似于 SQL 里的 "WHERE budget IN ('Low', 'Moderate')"
    filters = Filter.by_property('budget').contains_any(['Low', 'Moderate']),
    
    # limit: 限制最终返回最匹配的 4 条结果
    limit = 4
)

In [23]:
# 遍历查询结果中的所有对象，并将它们的属性打印出来
for obj in result.objects:
    # result.objects 是一个列表，包含了所有满足语义搜索和过滤条件的数据点
    # 每个 obj 是一个特定的 Weaviate 对象，包含了它的属性、元数据和唯一 ID
    
    # 使用自定义的辅助函数 print_object_properties 来展示内容
    # obj.properties：这是一个字典（Dict），存储了我们在创建集合时定义的字段数据
    # 例如：{'place': '哈尔滨', 'budget': 'Low', 'description': '...', ...}
    print_object_properties(obj.properties)
    
    # 这个函数会自动处理文本截断，确保你的控制台不会被超长正文“淹没”

last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Winter
place: Times Square
description: Bustling pedestrian intersection and major commercial hub.
state: New York
attractions: Broadway Theaters, New Year’s Eve Ball Drop
budget: Low
user_ratings: 4.3



place: Glacier National Park
best_season_to_visit: Summer
state: Montana
description: Park known for its rugged mountains and alpine forests.
last_updated: 2023-10-01 00:00:00+00:00
attractions: Going-to-the-Sun Road, Grinnell Glacier
budget: Moderate
user_ratings: 4.8



place: Zion National Park
state: Utah
last_updated: 2023-10-01 00:00:00+00:00
description: Beautiful park known for its impressive canyons and towering cliffs.
best_season_to_visit: Spring, Fall
attractions: The Narrows, Angels Landing
budget: Moderate
user_ratings: 4.7



last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Summer
state: Massachusetts
description: Popular tourist destination known for its beaches and quaint towns.
place: Cape Cod

<a id='3-3'></a>
### 3.3 BM25 search

To perform BM25 search, just run `colections.query.bm25`, the usual parameters `query`, `limit` and `filters` can be passed.

### 3.3 BM25搜索
要执行BM25搜索，只需运行 `colections.query.bm25`，通常的参数’查询‘,’限制‘和’过滤器'可以传递。

In [30]:
# 执行“BM25 关键词搜索 + 属性过滤”的复合查询
result = collection.query.bm25(
    # query: 关键词查询字符串
    # BM25 算法会寻找包含 "Winter"、"travel"、"cheap" 等具体单词的文档。
    # 它会根据词频（TF）和逆文档频率（IDF）来计算评分，单词匹配越精确、越罕见，得分越高。
    query = 'I want suggestions to travel during Winter. I want cheap places.', 
    
    # filters: 硬性过滤条件
    # 在进行关键词匹配的同时，依然强制要求 'budget' 字段必须是 'Low' 或 'Moderate'。
    filters = Filter.by_property('budget').contains_any(['Low', 'Moderate']),
    
    # limit: 限制返回结果的数量为 4 条
    limit = 4
)

In [31]:
# 遍历查询结果中的所有对象，并打印它们的属性
for obj in result.objects:
    # result.objects：包含了所有通过 BM25 算法匹配到的景点列表。
    # 在 BM25 模式下，这个列表是按照“关键词匹配得分”从高到低排列的。
    
    # 调用自定义的辅助函数 print_object_properties
    # obj.properties：这是存储在 Weaviate 中的原始业务数据字典。
    # 包含了如 'place'、'description'、'budget' 等字段。
    print_object_properties(obj.properties)
    
    # 提醒：该函数会自动处理文本截断，方便你在控制台快速预览多个结果。

place: Times Square
best_season_to_visit: Winter
last_updated: 2023-10-01 00:00:00+00:00
description: Bustling pedestrian intersection and major commercial hub.
state: New York
attractions: Broadway Theaters, New Year’s Eve Ball Drop
budget: Low
user_ratings: 4.3





<a id='3-4'></a>
### 3.4 混合搜索 (Hybrid Search)

这种搜索即你在课程中所见的 RRF 搜索。除了标准的查询参数外，你还可以传递一个 `alpha` 参数，用于控制混合中 BM25 搜索所占的比重。

In [32]:
# 执行“混合搜索 + 属性过滤”的复合查询
result = collection.query.hybrid(
    # query: 搜索字符串
    # 混合搜索会同时在“语义空间”和“关键词索引”中进行查找
    query = 'I want suggestions to travel during Winter. I want cheap places.', 
    
    # filters: 硬性过滤条件
    # 无论搜索结果的相关性如何，'budget' 必须符合 ['Low', 'Moderate'] 之一
    filters = Filter.by_property('budget').contains_any(['Low', 'Moderate']),
    
    # alpha: 混合权重参数（取值范围 0.0 到 1.0）
    # alpha = 1.0: 完全使用语义搜索 (nearText)
    # alpha = 0.0: 完全使用关键词搜索 (BM25)
    # alpha = 0.3: 此时权重偏向关键词匹配。
    # 作用：它会优先保证结果中包含 "Winter" 或 "Cheap" 单词，同时兼顾语义上的相关性。
    alpha = 0.3,
    
    # limit: 限制返回结果的数量为 4 条
    limit = 4
)

In [33]:
# 遍历混合搜索返回的结果对象列表
for obj in result.objects:
    # result.objects 是一个按综合评分从高到低排列的列表
    # 这个评分是向量相似度（Alpha部分）和 BM25 关键词得分（1-Alpha部分）融合后的结果
    # 融合算法通常采用倒数排名融合（RRF），确保了结果既“懂意思”又“中单词”
    
    # 调用自定义的辅助函数 print_object_properties
    # obj.properties：这是 Weaviate 中存储的核心业务字段字典
    # 它包含了 'place'、'description'、'budget' 等我们在创建集合时定义的属性
    print_object_properties(obj.properties)
    
    # 该函数会自动对长字段进行截断，让你能清晰地看到混合搜索推荐的前几个目的地

last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Winter
state: New York
description: Bustling pedestrian intersection and major commercial hub.
place: Times Square
attractions: Broadway Theaters, New Year’s Eve Ball Drop
budget: Low
user_ratings: 4.3



place: Glacier National Park
state: Montana
last_updated: 2023-10-01 00:00:00+00:00
description: Park known for its rugged mountains and alpine forests.
best_season_to_visit: Summer
attractions: Going-to-the-Sun Road, Grinnell Glacier
budget: Moderate
user_ratings: 4.8



last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Spring, Fall
state: Utah
description: Beautiful park known for its impressive canyons and towering cliffs.
place: Zion National Park
attractions: The Narrows, Angels Landing
budget: Moderate
user_ratings: 4.7



last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Summer
state: Massachusetts
description: Popular tourist destination known for its beaches and quaint towns.
place: Cape Cod

<a id='3-5'></a>
### 3.5 Reranking

You can easily perform reranking with Weaviate by passing a new argument to a search. Let's try with semantic search!

### 3.5重新排名
通过向搜索传递一个新参数，可以很容易地使用Weaviate执行重新排序。让我们尝试一下语义搜索！

In [34]:
# 从 Weaviate 的查询模块中导入 Rerank 类
from weaviate.classes.query import Rerank

# 执行“语义搜索 + 智能重排序”的复合查询
response = collection.query.near_text(
    # query: 初始语义搜索字符串
    # 这一步会利用向量索引，从成千上万条数据中快速捞出最相关的候选者
    query="'I want suggestions to travel during Winter. I want cheap and fun places.'",  
    
    # limit: 初始检索的数量
    # 注意：重排序是在这 5 个结果内部进行的，目的是调整它们的先后顺序
    limit=5,
    
    # rerank: 定义重排序逻辑（双阶段检索的核心）
    # 作用：调用更强大的 Cross-Encoder 模型对初始结果进行深度语义对比
    rerank=Rerank(
        # prop: 指定用于重排序的属性字段
        # 这里选择 "attractions"（景点），意味着重排模型会重点阅读景点的具体内容
        prop="attractions",                   
        
        # query: 用于重排序的特定目标
        # 即使初始查询包含冬天、便宜等多个信息，在重排阶段我们更关注“好玩（Fun places）”
        # 如果不提供此参数，默认会使用最上面的原始 query
        query="Fun places"  
    )
)

In [35]:
# 遍历重排序（Rerank）后的结果对象列表
for obj in result.objects:
    # 注意：如果上一单元格定义的变量名是 'response'，此处应确保变量名一致（如改为 response.objects）
    
    # result.objects 是经过 Cross-Encoder 模型深度打分后重新排列的列表
    # 哪怕在第一阶段向量搜索中排在第 5 名的景点，
    # 如果它的 'attractions' 描述极其符合“Fun places”，现在也可能被提拔到第 1 名。
    
    # 调用自定义的辅助函数 print_object_properties
    # obj.properties：包含该景点的核心数据（地名、状态、描述、季节、景点、预算等）
    print_object_properties(obj.properties)
    
    # 进阶提示：在重排序模式下，你还可以访问 obj.metadata.rerank_score
    # 这个分数代表了重排模型对该文档与“Fun places”匹配程度的深度认可。

last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Winter
state: New York
description: Bustling pedestrian intersection and major commercial hub.
place: Times Square
attractions: Broadway Theaters, New Year’s Eve Ball Drop
budget: Low
user_ratings: 4.3



place: Glacier National Park
state: Montana
last_updated: 2023-10-01 00:00:00+00:00
description: Park known for its rugged mountains and alpine forests.
best_season_to_visit: Summer
attractions: Going-to-the-Sun Road, Grinnell Glacier
budget: Moderate
user_ratings: 4.8



last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Spring, Fall
state: Utah
description: Beautiful park known for its impressive canyons and towering cliffs.
place: Zion National Park
attractions: The Narrows, Angels Landing
budget: Moderate
user_ratings: 4.7



last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Summer
state: Massachusetts
description: Popular tourist destination known for its beaches and quaint towns.
place: Cape Cod

In [ ]:
# Don't forget to close the client!
client.close()

: 

Keep it up! You just finished the basics of the Weaviate API you'll use in this course!

保持下去！您刚刚完成了将在本课程中使用的Weaviate API的基础知识！